# Preprocessing 서울시 지하철역 엘리베이터 위치정보

data load

In [1]:
import pandas as pd
import os

# File path
file_path = '../../data/raw/서울시 지하철역 엘리베이터 위치정보.csv'

# Load data
try:
    # Subway data often use cp949 or euc-kr encoding
    df = pd.read_csv(file_path, encoding='cp949')
    print("Data loaded successfully.")
    display(df.head())
except Exception as e:
    print(f"Error loading data: {e}")

Data loaded successfully.


,노드링크 유형,노드 WKT,노드 ID,노드 유형 코드,시군구코드,시군구명,읍면동코드,읍면동명,지하철역코드,지하철역명
0,NODE,POINT(127.01049296509703 37.571417966123946),212410,0,1111000000,종로구,1111017400,창신동,272,동대문
1,NODE,POINT(127.01551499790813 37.57951303710329),212418,0,1111000000,종로구,1111017500,숭인동,270,창신
2,NODE,POINT(126.98515892357797 37.57622646659184),211632,0,1111000000,종로구,1111013400,경운동,269,안국
3,NODE,POINT(127.01505874969273 37.57992200287952),167351,1,1111000000,종로구,1111017400,창신동,270,창신
4,NODE,POINT(126.97413365201709 37.575965675947124),212372,1,1111000000,종로구,1111010700,적선동,271,경복궁(정부서울청사)


중복데이터 확인

In [2]:
print(df['지하철역명'].value_counts().head(10))
print('\n전체 행:', len(df))
print('고유 역 수:', df['지하철역명'].nunique())

지하철역명
온수(성공회대입구)    7
삼각지           6
노원            6
강남            6
서울역           6
방학            5
신림            5
마곡나루          5
당산            5
신설동           5
Name: count, dtype: int64

전체 행: 552
고유 역 수: 271


한 역에 엘레베이터가 여러개면 => 역당 중복데이터 존재!  
**좌표 평균으로 처리** 할께요

In [3]:
import re

# WKT에서 위경도 추출
df['경도'] = df['노드 WKT'].str.extract(r'POINT\((\S+) \S+\)').astype(float)
df['위도'] = df['노드 WKT'].str.extract(r'POINT\(\S+ (\S+)\)').astype(float)

# 역별 좌표 평균
df_station = df.groupby(['지하철역코드', '지하철역명', '시군구코드', '시군구명']).agg(
    위도=('위도', 'mean'),
    경도=('경도', 'mean')
).reset_index()

print(df_station.shape)


(293, 6)


In [4]:
df_station['지하철역명'] = df_station['지하철역명'] + '역'

print(df_station['지하철역명'].head(10))

0       압구정역
1       학여울역
2      삼성중앙역
3        구룡역
4     대모산입구역
5        한티역
6        신사역
7        대청역
8    압구정로데오역
9        수서역
Name: 지하철역명, dtype: object


In [5]:
df_station.head(10)

,지하철역코드,지하철역명,시군구코드,시군구명,위도,경도
0,1,압구정역,1168000000,강남구,37.526456,127.028798
1,2,학여울역,1168000000,강남구,37.496481,127.070856
2,3,삼성중앙역,1168000000,강남구,37.513131,127.053637
3,4,구룡역,1168000000,강남구,37.486751,127.059762
4,5,대모산입구역,1168000000,강남구,37.491575,127.073233
5,6,한티역,1168000000,강남구,37.496353,127.053132
6,7,신사역,1168000000,강남구,37.516516,127.020517
7,8,대청역,1168000000,강남구,37.494052,127.079174
8,9,압구정로데오역,1168000000,강남구,37.528030,127.040779
9,10,수서역,1168000000,강남구,37.487288,127.101695


**저장**

In [ ]:
output_path = '../../data/preprocessed/지하철역_위치정보_전처리.csv'
df_station.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f"✅ 지하철역 전처리 데이터가 저장되었습니다: {output_path}")

✅ 지하철역 전처리 데이터가 저장되었습니다: ../../data/preprocessed/지하철역_위치정보_전처리.csv
